In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

## You need the tokenizer of your dataset 

In [ ]:
from data import TextDataloaderConfig, create_text_dataloaders

CFG = TextDataloaderConfig(
    preset_name="wikitext2",      # wikitext2, tinystories, ag_news, imdb, minipile
    block_size=128,
    batch_size=8,
    max_tokenizer_documents=5000,
    max_train_documents=2000,
    max_validation_documents=500,
    tokenizer_path="data/cache/tokenizers/wikitext2.json",
)

train_loader, val_loader, tokenizer = create_text_dataloaders(
    cfg=CFG,
    use_mtp=False,
)

## And define the model to load the weights 

In [ ]:
from src.mini_deepseek_class import * 

cfg = DeepSeekV4LMConfig(
    vocab_size=216,
    d_model=64,
    n_layers=4,
    max_seq_len=64,
    pad_token_id=0,
    ignore_index=-100,

    # ========================================================
    # Attention: hybrid CSA/HCA baseline
    # Pattern:
    #   layer 0 -> CSA
    #   layer 1 -> HCA
    #   layer 2 -> CSA
    #   layer 3 -> HCA
    # ========================================================
    attention_type="hybrid",
    attention_pattern=("csa", "hca"),

    n_heads=4,
    head_dim=16,
    rotary_dim=16,

    attention_dropout=0.0,
    residual_dropout=0.0,
    use_attention_bias=False,
    use_rope=True,
    rope_theta=10000.0,

    # ========================================================
    # CSA config
    # ========================================================
    compression_factor=4,
    top_k_blocks=2,
    window_size=8,
    indexer_dim=8,
    n_indexer_heads=2,
    query_compression_dim=16,

    use_attention_sink=True,
    use_grouped_output_projection=True,
    output_projection_groups=2,
    use_indexer_score_bias=False,
    use_separate_local_kv=True,

    # ========================================================
    # HCA config
    # HCA should compress more aggressively than CSA.
    # With max_seq_len=64, hca_compression_factor=8 is a sane
    # mini baseline. 16 also works but gives very few global blocks.
    # ========================================================
    hca_compression_factor=8,

    # ========================================================
    # FFN: DeepSeekMoE
    # ========================================================
    ffn_type="moe",
    num_experts=4,
    top_k_experts=2,

    expert_hidden_dim=128,
    expert_expansion_factor=4.0,
    expert_multiple_of=1,

    shared_experts=1,
    shared_hidden_dim=128,
    shared_expansion_factor=4.0,

    router_type="learned",
    router_score_fn="sqrt_softplus",
    normalize_topk_weights=True,
    topk_weight_scale=1.0,
    router_jitter_noise=0.0,
    hash_routing_stride=1,

    routed_scale=1.0,
    shared_scale=1.0,

    balance_loss_weight=0.01,
    sequence_balance_loss_weight=0.01,

    # ========================================================
    # mHC
    # ========================================================
    use_mhc=True,
    n_hc=4,
    mhc_sinkhorn_iters=20,
    mhc_eps=1e-6,
    mhc_dynamic=True,
    mhc_expand_mode="first",
    mhc_collapse_mode="readout",

    mhc_use_log_sinkhorn=False,
    mhc_sinkhorn_fp32=True,
    mhc_init_alpha=1e-3,
    mhc_alpha_max=1.0,
    mhc_bounded_alpha=True,

    # ========================================================
    # MTP
    # ========================================================
    use_mtp=True,
    mtp_depth=2,
    mtp_hidden_dim=64,
    use_mtp_transform=True,
    mtp_activation="silu",
    mtp_dropout=0.0,
    mtp_loss_weight=0.3,
    mtp_tie_with_lm_head=False,
    mtp_depth_loss_weights=None,
    mtp_validate_label_range=True,

    # ========================================================
    # Embedding / norm / init
    # ========================================================
    embedding_dropout=0.0,
    scale_embeddings=False,
    tie_word_embeddings=True,

    rms_norm_eps=1e-6,
    init_std=0.02,

    # ========================================================
    # Loss semantics
    #   input_ids = ids[:-1]
    #   labels    = ids[1:]
    # ========================================================
    labels_are_shifted=True,
    ignore_pad_token_in_loss=True,)

model = DeepSeekV4LM(cfg)
model.to('cuda')


## Inference 

In [ ]:
from inference import audit_inference_pipeline, inference_autoregresive

out = inference_autoregresive(
    model,
    prompt="", # Promt 
    tokenizer=tokenizer, # Tokenizer of your data 
    cache_mode="audit",          # use "mha_decode" only for pure MHA models
    max_new_tokens=32,
    do_sample=True,
    temperature=0.8,
    top_k=20,
    top_p=0.95,
    repetition_penalty=1.05,
    eos_token_id=tokenizer.eos_id,
    pad_token_id=tokenizer.pad_id,
    use_mtp_draft=True,
    max_mtp_draft_tokens=2,
    return_cache_stats=True,
)

print(out["text"])
print(out["cache_stats"])


In [ ]:
out = inference_autoregresive(
    model,
    prompt="", #Promt
    tokenizer=tokenizer,
    cache_mode="deepseek_decode",
    max_new_tokens=32,
    do_sample=False,
    use_mtp_draft=True,
    return_cache_stats=True,
)

print(out["text"])
print(out["cache_stats"])

